In [ ]:
import os
from pathlib import Path

# Force a stable W&B directory across sessions/devices
WANDB_DIR = Path("/home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/wandb")
os.environ["WANDB_DIR"] = str(WANDB_DIR)
WANDB_DIR.mkdir(parents=True, exist_ok=True)


# Epoch 3 Sample Comparison (Qwen3 Abstract Evaluator)

This notebook:
- Recreates the same deterministic split logic from your training notebook.
- Loads the saved `epoch_3` adapter.
- Runs inference on 5 samples from the same test split.
- Compares original vs predicted score and rationale.


In [ ]:
import json
import re
import random
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

from unsloth import FastLanguageModel

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

OUTPUT_ROOT = Path("../artifacts/abstract_evaluator_qwen3_sft")
DATA_PATH = Path("../../data/processed/final_sft_dataset_1220.jsonl")
MODEL_NAME = "Qwen/Qwen3-8B"
MAX_SEQ_LENGTH = 2048

TRAIN_ROWS = 910
VAL_ROWS = 100
TEST_ROWS = 100

EPOCH3_ADAPTER_DIR = OUTPUT_ROOT / "models" / "qwen3_8b_abstract_evaluator_lora_no_quant" / "epoch_adapters" / "epoch_3"
assert EPOCH3_ADAPTER_DIR.exists(), f"Missing adapter dir: {EPOCH3_ADAPTER_DIR}"
print("Using adapter:", EPOCH3_ADAPTER_DIR)


In [ ]:
REQUIRED_COLUMNS = ["paper_id", "submission", "score", "rationale"]

DEFAULT_TASK = "Evaluate the quality of the following research abstract for conference acceptance."
DEFAULT_REFERENCE = "A strong research abstract clearly presents the problem, methodology, contribution, and experimental evidence."
DEFAULT_RUBRIC = {
    "score_scale": {
        "0": "Very poor abstract: missing most core components, unclear, generic, or unusable.",
        "1": "Weak abstract: contains a few useful elements but major components are missing or vague.",
        "2": "Borderline abstract: understandable but incomplete; some important components are weak or missing.",
        "3": "Good abstract: mostly complete, clear, and logically structured, with minor weaknesses.",
        "4": "Excellent abstract: complete, clear, concise, well-structured, and strongly communicates the paper's contribution and evidence."
    },
    "criteria": {
        "1_background_or_context": "Provides enough background or introduction to understand the research area and motivation.",
        "2_problem_statement": "Clearly identifies the research problem, gap, or limitation being addressed.",
        "3_objective_or_purpose": "States the main objective, research question, or purpose of the work.",
        "4_methodology": "Explains the methods, approach, model, experiment, dataset, or procedure used.",
        "5_results_or_findings": "Reports concrete results, findings, observations, or evidence rather than only intentions.",
        "6_contribution": "Clarifies what is new, useful, or significant about the work.",
        "7_conclusion_or_implication": "Provides a conclusion, implication, impact, or takeaway from the work.",
        "8_clarity_and_conciseness": "Uses clear, precise, and concise language without unnecessary vagueness or filler.",
        "9_logical_flow": "Presents the abstract in a coherent order: context/problem -> objective -> method -> results -> contribution.",
        "10_specificity_and_precision": "Includes specific details (e.g., methods, data, outcomes) rather than broad generic claims."
    }
}

def load_final_dataframe(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"DATA_PATH does not exist: {path}")
    if path.suffix.lower() == ".jsonl":
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    return pd.read_json(path)

def format_rubric(rubric: Dict[str, Any]) -> str:
    if isinstance(rubric, dict) and "score_scale" in rubric and "criteria" in rubric:
        score_scale = rubric.get("score_scale", {})
        criteria = rubric.get("criteria", {})
        score_lines = ["Score scale:"]
        for k, v in sorted(score_scale.items(), key=lambda x: int(str(x[0]))):
            score_lines.append(f"{k} = {v}")
        criteria_lines = ["Criteria:"]
        for k, v in sorted(criteria.items(), key=lambda x: str(x[0])):
            criteria_lines.append(f"{k}: {v}")
        return "\n".join(score_lines + [""] + criteria_lines)
    return "\n".join([f"{k} = {v}" for k, v in sorted(rubric.items(), key=lambda x: str(x[0]))])

def make_user_prompt(row: pd.Series) -> str:
    title_block = ""
    if "title" in row and pd.notna(row["title"]) and str(row["title"]).strip():
        title_block = f"\nTitle:\n{str(row['title']).strip()}\n"
    return (
        "/no_think\n"
        f"Task:\n{row['task']}\n\n"
        f"Reference:\n{row['reference']}\n\n"
        f"Rubric:\n{format_rubric(row['rubric'])}\n"
        f"{title_block}\n"
        f"Submission:\n{row['submission']}\n\n"
        "Return only valid JSON with exactly these keys: score, rationale.\n"
        "Do not include markdown, analysis, or extra text."
    )

def make_messages(row: pd.Series) -> List[Dict[str, str]]:
    return [
        {"role": "system", "content": "You are a strict research abstract evaluator. You return only valid JSON."},
        {"role": "user", "content": make_user_prompt(row)},
        {"role": "assistant", "content": json.dumps({"score": int(row["score"]), "rationale": str(row["rationale"]).strip()}, ensure_ascii=False)},
    ]


In [ ]:
def _pick_degradation_col(df: pd.DataFrame) -> Optional[str]:
    for c in ["degradation_type", "degradation", "degrade_type", "perturbation_type"]:
        if c in df.columns:
            return c
    return None

def _build_joint_strata(df: pd.DataFrame, score_col: str = "score", deg_col: Optional[str] = None) -> pd.Series:
    score_part = df[score_col].astype(str)
    if deg_col is None:
        return score_part
    deg_part = df[deg_col].fillna("missing").astype(str)
    joint = score_part + "||" + deg_part
    counts = joint.value_counts()
    rare = counts[counts < 2].index
    if len(rare) > 0:
        joint = joint.where(~joint.isin(rare), score_part + "||__other__")
    if joint.value_counts().min() < 2:
        return score_part
    return joint

def split_fixed_counts_stratified(df: pd.DataFrame, train_rows=TRAIN_ROWS, val_rows=VAL_ROWS, test_rows=TEST_ROWS, seed=SEED):
    total_needed = train_rows + val_rows + test_rows
    if len(df) < total_needed:
        raise ValueError(f"Need at least {total_needed} rows, found {len(df)}")

    base = df.sample(frac=1.0, random_state=seed).head(total_needed).reset_index(drop=True)

    deg_col = _pick_degradation_col(base)
    strata_all = _build_joint_strata(base, score_col="score", deg_col=deg_col)

    try:
        rest_df, test_df = train_test_split(base, test_size=test_rows, random_state=seed, stratify=strata_all)
    except ValueError:
        rest_df, test_df = train_test_split(base, test_size=test_rows, random_state=seed, stratify=base["score"])

    val_rows = int(val_rows)
    try:
        strata_rest = _build_joint_strata(rest_df, score_col="score", deg_col=deg_col if deg_col in rest_df.columns else None)
        train_df, val_df = train_test_split(rest_df, test_size=val_rows, random_state=seed, stratify=strata_rest)
    except ValueError:
        train_df, val_df = train_test_split(rest_df, test_size=val_rows, random_state=seed, stratify=rest_df["score"])

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


df = load_final_dataframe(DATA_PATH)
for col in REQUIRED_COLUMNS:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

if "task" not in df.columns:
    df["task"] = DEFAULT_TASK
if "reference" not in df.columns:
    df["reference"] = DEFAULT_REFERENCE
if "rubric" not in df.columns:
    df["rubric"] = [DEFAULT_RUBRIC] * len(df)

train_df, val_df, test_df = split_fixed_counts_stratified(df)
for part in [train_df, val_df, test_df]:
    part["messages"] = part.apply(make_messages, axis=1)

print("Split sizes:", len(train_df), len(val_df), len(test_df))


In [ ]:
def _normalize_messages_for_template(messages):
    if isinstance(messages, dict):
        roles = messages.get("role", [])
        contents = messages.get("content", [])
        if isinstance(roles, list) and isinstance(contents, list):
            return [{"role": str(r), "content": str(c)} for r, c in zip(roles, contents)]
    if isinstance(messages, list):
        out = []
        for m in messages:
            if isinstance(m, dict):
                out.append({"role": str(m.get("role", "user")), "content": str(m.get("content", ""))})
        return out
    return [{"role": "user", "content": str(messages)}]

def apply_qwen3_chat_template(tokenizer, messages, add_generation_prompt=False):
    messages = _normalize_messages_for_template(messages)
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt)

def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    if not isinstance(text, str):
        return None
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    match = re.search(r"\{.*?\}", text, flags=re.S)
    if match:
        try:
            obj = json.loads(match.group(0))
            if isinstance(obj, dict):
                return obj
        except Exception:
            return None
    return None

def parse_score(text: str) -> Optional[int]:
    obj = extract_json_object(text)
    if obj is not None and "score" in obj:
        try:
            score = int(obj["score"])
            if 0 <= score <= 4:
                return score
        except Exception:
            pass
    match = re.search(r'"?score"?\s*[:=]\s*([0-4])', str(text))
    if match:
        return int(match.group(1))
    return None

def parse_rationale(text: str) -> str:
    obj = extract_json_object(text)
    if obj is not None and "rationale" in obj:
        return str(obj["rationale"]).strip()
    return str(text).strip()

def make_inference_prompt(tokenizer, messages):
    system_user_messages = [m for m in messages if m["role"] in ["system", "user"]]
    return apply_qwen3_chat_template(tokenizer, system_user_messages, add_generation_prompt=True)

@torch.no_grad()
def generate_predictions(eval_df: pd.DataFrame, model, tokenizer, max_new_tokens=180, batch_size=5):
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    prompts = [make_inference_prompt(tokenizer, msgs) for msgs in eval_df["messages"]]
    predictions = []

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        prompt_len = inputs["input_ids"].shape[1]
        decoded = tokenizer.batch_decode(outputs[:, prompt_len:], skip_special_tokens=True)
        predictions.extend(decoded)

    out = eval_df.copy().reset_index(drop=True)
    out["prediction_text"] = predictions
    out["pred_score"] = out["prediction_text"].apply(parse_score)
    out["pred_rationale"] = out["prediction_text"].apply(parse_rationale)
    out["score_match"] = out["score"].astype(int) == out["pred_score"].fillna(-999).astype(int)
    return out



In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(EPOCH3_ADAPTER_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16,
    load_in_4bit=False,
)

# Decoder-only generation should use left padding for batched prompts.
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

FastLanguageModel.for_inference(model)
model.generation_config.pad_token_id = tokenizer.pad_token_id

print("Loaded epoch_3 adapter with left-padding tokenizer for inference.")



In [ ]:
sample_df = test_df.sample(n=5, random_state=SEED).reset_index(drop=True)
pred_df = generate_predictions(sample_df, model=model, tokenizer=tokenizer, max_new_tokens=180, batch_size=5)

compare_cols = [
    "paper_id",
    "score",
    "pred_score",
    "score_match",
    "rationale",
    "pred_rationale",
]

pd.set_option("display.max_colwidth", 240)
pred_df[compare_cols]


In [ ]:
out_path = OUTPUT_ROOT / "eval" / "qwen3_8b_abstract_evaluator_lora_no_quant" / "epoch_3_test5_comparison.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
pred_df.to_csv(out_path, index=False)
print("Saved:", out_path)
